# M23 · RLHF, DPO, PPO

_Curriculum · Domain 5 · Reinforcement learning_

**Tune an assistant from preference signals without forgetting the reference model.**

This notebook builds the smallest useful RLHF playground: Bradley-Terry preferences, a KL-penalized policy objective, a PPO clipped term, and a DPO-style direct preference update.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(23)

## Preference probability

A reward model often starts from pairwise labels. If response $w$ beats response $l$, the Bradley-Terry model uses

$$P(w \succ l)=\sigma(r_w-r_l).$$

Only the reward difference matters.

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

reward_winner = 1.4
reward_loser = 0.2
preference_prob = sigmoid(reward_winner - reward_loser)

print(round(preference_prob, 4))
assert 0.76 < preference_prob < 0.77

## Tiny prompt-tuning policy

Pretend one Creative Intelligence prompt has three candidate completions. The reference model is safe but generic; the trainable policy shifts probability toward the preferred completion while paying a KL cost.

In [ ]:
actions = np.array(["plain", "specific", "risky"])
ref_logits = np.array([0.2, 0.1, -0.4])
policy_logits = np.array([0.0, 0.4, -0.5])
rewards = np.array([0.1, 1.0, -0.2])

def softmax(z):
    shifted = z - np.max(z)
    weights = np.exp(shifted)
    return weights / weights.sum()

ref_probs = softmax(ref_logits)
policy_probs = softmax(policy_logits)

print(pd.DataFrame({"action": actions, "ref": ref_probs, "policy": policy_probs, "reward": rewards}))
assert np.isclose(policy_probs.sum(), 1.0)

## KL-penalized objective

A simple RLHF objective is expected reward minus a reference penalty:

$$J(\pi)=\sum_a \pi(a) r(a)-\beta\sum_a \pi(a)\log\frac{\pi(a)}{\pi_0(a)}.$$

The penalty keeps the tuned model near the model we trust.

In [ ]:
beta = 0.2
expected_reward = np.sum(policy_probs * rewards)
kl_to_ref = np.sum(policy_probs * np.log(policy_probs / ref_probs))
objective = expected_reward - beta * kl_to_ref

print("expected reward", round(expected_reward, 4))
print("kl to ref", round(kl_to_ref, 4))
print("objective", round(objective, 4))
assert objective < expected_reward

## PPO clipped ratio term

PPO updates from logged samples. For one sampled action, it compares the new probability to the old probability with $\rho=\pi_{new}(a)/\pi_{old}(a)$ and clips the ratio before multiplying by the advantage.

In [ ]:
old_prob = 0.30
new_prob = 0.42
advantage = 0.80
epsilon = 0.20
ratio = new_prob / old_prob
clipped_ratio = np.clip(ratio, 1.0 - epsilon, 1.0 + epsilon)
unclipped = ratio * advantage
clipped = clipped_ratio * advantage
ppo_term = min(unclipped, clipped)

print("ratio", round(ratio, 3))
print("unclipped", round(unclipped, 3))
print("clipped", round(clipped, 3))
print("ppo term", round(ppo_term, 3))
assert np.isclose(ppo_term, 0.96)

## DPO-style direct preference signal

DPO skips an explicit reward model. It compares the policy log-ratio between winner and loser against the reference log-ratio, then pushes the winner up when the policy is not sufficiently better than reference.

In [ ]:
chosen = 1
rejected = 0
policy_log_ratio = np.log(policy_probs[chosen]) - np.log(policy_probs[rejected])
ref_log_ratio = np.log(ref_probs[chosen]) - np.log(ref_probs[rejected])
beta_dpo = 0.5
margin = beta_dpo * (policy_log_ratio - ref_log_ratio)
dpo_prob = sigmoid(margin)

print("dpo preference probability", round(dpo_prob, 4))
assert 0.5 < dpo_prob < 0.6

## Compare update families

The same data can support three views: learn a reward model from preferences, optimize with PPO under a KL guardrail, or optimize preferences directly with DPO.

In [ ]:
names = ["BT preference", "KL objective", "PPO term", "DPO prob"]
values = [preference_prob, objective, ppo_term, dpo_prob]

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(names, values, color="#4c78a8")
ax.set_title("tiny RLHF calculations")
ax.set_ylim(0.0, 1.1)
plt.xticks(rotation=20)
plt.show()